# Classifying Particles with a Convolutional Neural Network


<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<a href="https://colab.research.google.com/github/Leolinen/TIF360/blob/main/HW2/particles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<strong>If using Colab/Kaggle:</strong> You need to uncomment the code in the cell below this one.
</div>

In [ ]:
!pip install deeplay  # Uncomment if using Colab/Kaggle.

In [ ]:
import pickle
import torch
import torch.utils.data
import matplotlib.pyplot as plt
import numpy as np
import deeplay as dl
import torchmetrics as tm
from torch.nn.functional import relu
from skimage.exposure import rescale_intensity
from skimage.transform import resize


## Loading the Particle Dataset


In [ ]:
with open("simple_particle_dataset.pkl", "rb") as f:
    raw = pickle.load(f)


class ParticleDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels):
        self.images = [
            torch.tensor(img, dtype=torch.float32).unsqueeze(0)
            for img in images
        ]
        self.labels = [
            torch.tensor(float(lbl[0]), dtype=torch.float32).unsqueeze(-1)
            for lbl in labels
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


dataset = ParticleDataset(raw["images"], raw["labels"])


### Splitting the Dataset and Defining the Data Loaders

In [ ]:
train, test = torch.utils.data.random_split(dataset, [0.8, 0.2])

... and define the data loaders.

In [ ]:
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=256, shuffle=False)

### Plotting the ROC Curve

Implement a function to plot the ROC curve ...

In [ ]:
def plot_roc(classifier, loader):
    """Plot ROC curve."""
    roc = tm.ROC(task="binary")
    f1 = tm.F1Score(task="binary")
    for image, label in loader:
        preds = classifier(image)
        roc.update(preds, label.long())
        f1.update(preds, label.long())

    fig, ax = roc.plot(score=True)
    ax.grid(False)
    ax.axis("square")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="center right")
    plt.show()

    print(f"F1 Score: {f1.compute():.4f}")


## Classifying the Particles with Convolutional Neural Networks

Implement a convolutional neural network with a dense top ...

In [ ]:
conv_base = dl.ConvolutionalNeuralNetwork(
    in_channels=1, hidden_channels=[16, 16, 32], out_channels=32,
)
conv_base.blocks[2].pool.configure(torch.nn.MaxPool2d, kernel_size=2)

connector = dl.Layer(torch.nn.AdaptiveAvgPool2d, output_size=1)

dense_top = dl.MultiLayerPerceptron(
    in_features=32, hidden_features=[], out_features=1,
    out_activation=torch.nn.Sigmoid,
)

cnn = dl.Sequential(conv_base, connector, dense_top)


... print out its detailed architecture ...

In [ ]:
print(cnn)

... compile it ...

In [ ]:
cnn_classifier = dl.BinaryClassifier(
    model=cnn, optimizer=dl.RMSprop(lr=0.001),
).create()

... and print out the compiled convolutional neural network.

In [ ]:
print(cnn_classifier)

### Training the Convolutional Neural Network

In [ ]:
cnn_trainer = dl.Trainer(max_epochs=5, accelerator="auto")
cnn_trainer.fit(cnn_classifier, train_loader)

### Testing the Convolutional Neural Network

In [ ]:
cnn_trainer.test(cnn_classifier, test_loader)

### Plotting the ROC Curve

In [ ]:
plot_roc(cnn_classifier, test_loader)

### Checking the Values of the Filters

The following code accesses the 32nd filter's weights in the first convolutional layer of the CNN. It navigates through the model's first module (`model[0]`), selects the initial block (`blocks[0]`), and then targets the layer's weights (`layer.weight[15]`).

In [ ]:
filter = cnn_classifier.model[0].blocks[0].layer.weight[15]

print(filter)

### Visualizing the Activations of the Convolutional Layers

Pick the image of an particle to then check the activations it produces on the last convolutional layer ...

In [ ]:
im_ind = 0
image, label = dataset[im_ind]
image_hr = image[0].numpy()  # (64, 64) grayscale


... verify whether this image is of a particle ...

In [ ]:
print(label)


... define a function to visualize the activations ...

In [ ]:
def plot_activations(activations, cols=8):
    """Visualize activations."""
    rows = -(activations.shape[0] // -cols)

    fig, axs = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows))
    for i, ax in enumerate(axs.ravel()):
        ax.axis("off")
        if i < activations.shape[0]:
            ax.imshow(activations[i].numpy())
            ax.set_title(i, fontsize=16)
    plt.show()

... add a hook to access the activations in the forward pass ...


In [ ]:
def hook_func(layer, input, output):
    """Hook for activations."""
    activations = output.detach().clone()
    plot_activations(activations[0])

for block in cnn_classifier.model[0].blocks:
    layer = block.layer
    handle_hook = layer.register_forward_hook(hook_func)

    try:
        pred = cnn_classifier.model(image.unsqueeze(0))
    except Exception as e:
        print(f"An error occurred during model prediction: {e}")
    finally:
        handle_hook.remove()

### Visualizing the Heatmaps

Use hooks to keep tracks also of the gradients in the backward pass ...

In [ ]:
hookdata = {}

def fwd_hook_func(layer, input, output):
    """Forward hook function."""
    hookdata["activations"] = output.detach().clone()

def bwd_hook_func(layer, grad_input, grad_output):
    """Backward hook function."""
    hookdata["gradients"] = grad_output[0].detach().clone()

layer = cnn_classifier.model[0].blocks[3].layer
handle_fwd_hook = layer.register_forward_hook(fwd_hook_func)
handle_bwd_hook = layer.register_full_backward_hook(bwd_hook_func)

try:
    pred = cnn_classifier.model(image.unsqueeze(0))
    pred.sum().backward()
except Exception as e:
    print(f"An error occurred during model prediction: {e}")
finally:
    handle_fwd_hook.remove()
    handle_bwd_hook.remove()

... calculate the heatmap combining activations and gradients ...

In [ ]:
activations = hookdata["activations"][0]
gradients = hookdata["gradients"][0]

pooled_gradients = gradients.mean(dim=[1, 2], keepdim=True)
heatmap = relu((pooled_gradients * activations).sum(0)).detach().numpy()


... and plot the heatmap.

In [ ]:
rescaled_image = rescale_intensity(image_hr, out_range=(0, 1))
resized_heatmap = resize(heatmap, rescaled_image.shape, order=2)
rescaled_heatmap = rescale_intensity(resized_heatmap, out_range=(0.25, 1))

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(rescaled_image, cmap="gray", interpolation="bilinear")
plt.title("Original image", fontsize=16)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(rescaled_heatmap, interpolation="bilinear")
plt.title("Heatmap with Grad-CAM", fontsize=16)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(rescaled_image * rescaled_heatmap, cmap="gray", interpolation="bilinear")
plt.title("Overlay", fontsize=16)
plt.axis("off")

plt.show()
